# Cantus Task Template

Build your first agent on Cantus in five cells. Open this notebook in Colab, hit *Run all*, and follow the inline guidance.

- **Cell 1** mounts Google Drive (one-time OAuth).
- **Cell 2** picks a `model_variant`, pins a `cantus_version`, installs the framework, and loads the model from your Drive path.
- **Cell 3** is where you write your `@skill`, `@analyzer`, `@validator`, `@workflow`, and `Memory` subclass.
- **Cell 4** runs `Agent.run` with your query.
- **Cell 5** opens the Inspector to replay the EventStream.

> Need to download the Gemma 4 weights first? Run `notebooks/admin_setup.ipynb` once to mirror `google/gemma-4-E2B-it` and `google/gemma-4-E4B-it` into a Drive directory you choose.

## Cell 1: mount Google Drive

The first run triggers a Colab OAuth prompt. Once mounted, your files appear under `/content/drive/MyDrive/` and `/content/drive/Shareddrives/`.

In [ ]:
#@title Mount Drive { display-mode: "form" }
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted at /content/drive")
print("Next: run Cell 2 to install Cantus and load the model.")


## Cell 2: pick variant, set path, install framework, load model

The form on the right lets you switch model variant, pin a Cantus version, and point at the Drive directory where the Gemma 4 weights live. Fill in `model_root` with the path you (or your administrator) used in `notebooks/admin_setup.ipynb`.

In [ ]:
#@title Setup { display-mode: "form" }
import os

#@markdown **Backbone model variant**:
#@markdown <br>
#@markdown <small>※ <code>E2B</code> is experimental — see docs/cookbook/errors.md (sub-3B models occasionally short-circuit an empty <code>final_answer</code> under grammar-constrained decoding; Cantus retries automatically, but if you see three or more consecutive <code>ValidationErrorObservation(validator_name=&quot;non_empty_final_answer&quot;, ...)</code> entries in a single run, switch back to E4B).</small>
# E2B: experimental — see docs/cookbook/errors.md
model_variant = "E4B" #@param ["E4B", "E2B"]

#@markdown **Cantus framework version** (git tag / branch / commit sha; pin a tag for reproducibility):
cantus_version = "v0.1.3" #@param {type:"string"}

#@markdown **Drive directory holding the model weights** (must contain `gemma-4-E4B-it-4bit/` and `gemma-4-E2B-it-4bit/` subdirectories — populate via `notebooks/admin_setup.ipynb`):
model_root = "/content/drive/MyDrive/cantus_models" #@param {type:"string"}

#@markdown ---
#@markdown <small>Common path patterns:<br>
#@markdown ・MyDrive (personal): <code>/content/drive/MyDrive/cantus_models</code><br>
#@markdown ・Shared Drive (team / class): <code>/content/drive/Shareddrives/&lt;your-shared-drive&gt;/models</code><br>
#@markdown <br>
#@markdown <code>cantus_version</code> accepts:<br>
#@markdown ・<b>tag</b> (e.g. <code>v0.1.3</code>, default) — pin a fixed version so the framework will not drift mid-session.<br>
#@markdown ・<b>branch</b> (e.g. <code>main</code>) — follow the latest commit.<br>
#@markdown ・<b>commit sha</b> — pin precisely for bug reproduction.</small>

# Gemma 4 needs a recent transformers (model_type "gemma4"); upgrade in case Colab pre-installed an older release.
%pip install -q -U transformers bitsandbytes accelerate

# Install Cantus directly from GitHub (no Drive snapshot needed).
# Use %pip (IPython magic) rather than !pip so the install lands in the same Python kernel.
%pip install -q "git+https://github.com/schola-cantorum/cantus@{cantus_version}"

from cantus import (
    skill, analyzer, validator, workflow, debug,
    Skill, Analyzer, Validator, Workflow, Memory,
    Result, Agent, Inspector,
    mount_drive_and_load,
)

# Load the model (drive_root is passed explicitly so we override the framework default Shared Drive path).
model_handle = mount_drive_and_load(variant=model_variant, drive_root=model_root)
print(f"Loaded {model_variant} from {model_root}")


### Cell 2 note: small-model E2B retry behaviour

When you select `E2B`, three things are worth knowing (Cantus v0.1.2+ runtime behaviour, full details in `docs/cookbook/errors.md`):

1. A single `agent.run` invocation may produce one or more `ValidationErrorObservation(validator_name="non_empty_final_answer", ...)` entries inside the EventStream. The 2B model is prone to short-circuiting an empty `final_answer` under grammar-constrained decoding; Cantus enforces non-empty answers at both the schema and runtime layers and retries automatically.
2. These entries are **not a bug** — they are the spec-defined retry behaviour. The contract lives in the `agent-runtime` Requirement "FinalAnswerAction.answer is non-empty".
3. If three or more consecutive `non_empty_final_answer` entries appear in a single `agent.run`, switch `model_variant` back to `E4B`. The 4B model is significantly more compliant under grammar-constrained tool calling. As a softer fallback you may pass `agent.run(query, max_iterations=12)` to give E2B more retry headroom, but switching to E4B remains the more reliable path.


## Cell 3: write your protocols

In [ ]:
# ---------- your @skill ----------
# @skill
# def your_skill(arg: str) -> str:
#     """Describe skill behaviour.
#
#     Args:
#         arg: argument description.
#     """
#     ...

# ---------- your @analyzer ----------
# @analyzer
# def your_analyzer(text: str):
#     ...

# ---------- your @validator ----------
# @validator
# def your_validator(value) -> Result:
#     return Result.success(value)

# ---------- your @workflow ----------
# @workflow
# def your_workflow(query: str):
#     ...

# ---------- your Memory (class-first only) ----------
# class YourMemory(Memory):
#     ...

## Cell 4: run the agent

In [ ]:
agent = Agent(model=model_handle)
result = agent.run("Replace this string with your own query")
print(result)

## Cell 5: open the Inspector (advanced)

In [ ]:
# Example: wrap one of your @skill definitions with @debug:
# @debug
# @skill
# def your_skill(...): ...
# Then re-run Cell 3 — stdout will print [debug] lines for each call.

Inspector(result.stream).replay()